# 01 — SCUT Passenger EV Telemetry Audit

**HealthCar · Phase 2: Data Acquisition & Exploratory Audit**

This notebook loads a real passenger EV dataset **after you run** `python scripts/download_data.py --only scut1`. It does not claim any real future-fault labels or trained models. The default reads at most 250,000 rows to manage laptop memory.

**Goals:** confirm the schema, inspect missing data and timestamps, analyze measured signals, and derive conservative monitoring features.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Open Jupyter from the repository and keep the notebook within notebooks/."
sys.path.insert(0, str(ROOT / "src"))
from healthcar.audit import read_dataset, build_features, audit, write_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)

## 1. Read official downloaded telemetry

If missing, run `python scripts/download_data.py --only scut1` in a terminal. Change `KIND = "scut7"` only after downloading the optional larger archive.

In [ ]:
KIND = "scut1"
MAX_ROWS = 250_000  # use None only if memory permits
df, source, candidates = read_dataset(KIND, MAX_ROWS, ROOT / "data" / "raw")
print("Source:", source)
print("Loaded shape:", df.shape)
display(df.head())
display(pd.DataFrame(candidates[:5]))

## 2. Data quality — no model accuracy claims

We audit **loaded rows only**, not automatically the entire month of measurements.

In [ ]:
report, checked = write_report(KIND, MAX_ROWS,
    raw_root=ROOT / "data" / "raw",
    reports_dir=ROOT / "data" / "reports")
print("Warnings:")
for warning in report["warnings"]:
    print("-", warning)
display(pd.DataFrame({"column": df.columns,
                      "dtype": [str(t) for t in df.dtypes],
                      "missing_percent": [report["missing_percent"].get(c) for c in df.columns]}))
print("Saved JSON/CSV audit to data/reports/")

## 3. Timing and operating conditions

Different SCUT vehicles have different nominal recording frequencies. Inspect gaps before computing rolling or rate-based features.

In [ ]:
if "Time" in df.columns:
    parsed = pd.to_datetime(df["Time"], errors="coerce")
    step = parsed.diff().dt.total_seconds()
    print("Parsed timestamps:", parsed.notna().mean())
    print("Negative time jumps:", int((step < 0).sum()))
    display(step[step > 0].describe(percentiles=[.5,.9,.95,.99]))
if "charging_signal" in df.columns:
    display(df["charging_signal"].value_counts(dropna=False).rename("records").to_frame())
print("Operating mode definitions must be checked against the publisher.")

## 4. Battery monitoring features

Voltage and temperature spreads are observable proxies, **not** verified damage or SOH. Pack power sign is unverified until checking the current convention.

In [ ]:
features = build_features(df)
derived = [c for c in ["cell_voltage_spread_V","cell_temperature_spread_C","battery_power_W_unverified_sign"] if c in features]
display(features[derived].describe(percentiles=[.01,.5,.95,.99]).T)
for c in derived:
    sample = pd.to_numeric(features[c], errors="coerce").replace([np.inf,-np.inf], np.nan).dropna()
    if len(sample):
        plt.figure(figsize=(9,3))
        plt.plot(sample.iloc[:min(3000, len(sample))].to_numpy(), linewidth=.7)
        plt.title(f"{c} — first {min(3000,len(sample))} valid records")
        plt.xlabel("Observation index (not uniformly spaced time)")
        plt.ylabel(c)
        plt.tight_layout()
        plt.show()

## 5. Next research decisions

After real dataset inspection, write answers in the accompanying `docs/evaluation_plan.md`: missingness and gap severity, source units and operating codes, whether robust rolling baselines are appropriate, and which anomaly thresholds would be defensible. **Do not use unlabeled SCUT data to claim future failure prediction accuracy.**